In [1]:
import kagglehub
import pandas as pd
import numpy as np

d:\Projetos\Machine-learning-exercises\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 01. Download do dataset e construção do exercício

In [2]:
# Download latest version
path = kagglehub.dataset_download("yasserh/loan-default-dataset")

df = pd.read_csv(path + "/Loan_Default.csv")
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 148670 entries, 0 to 148669
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   ID                         148670 non-null  int64  
 1   year                       148670 non-null  int64  
 2   loan_limit                 145326 non-null  str    
 3   Gender                     148670 non-null  str    
 4   approv_in_adv              147762 non-null  str    
 5   loan_type                  148670 non-null  str    
 6   loan_purpose               148536 non-null  str    
 7   Credit_Worthiness          148670 non-null  str    
 8   open_credit                148670 non-null  str    
 9   business_or_commercial     148670 non-null  str    
 10  loan_amount                148670 non-null  int64  
 11  rate_of_interest           112231 non-null  float64
 12  Interest_rate_spread       112031 non-null  float64
 13  Upfront_charges            109028 non-nu

In [4]:
df.describe()

,ID,year,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,property_value,income,Credit_Score,LTV,Status,dtir1
count,148670.000000,148670.0,1.486700e+05,112231.000000,112031.000000,109028.000000,148629.000000,1.335720e+05,139520.000000,148670.000000,133572.000000,148670.000000,124549.000000
mean,99224.500000,2019.0,3.311177e+05,4.045476,0.441656,3224.996127,335.136582,4.978935e+05,6957.338876,699.789103,72.746457,0.246445,37.732932
std,42917.476598,0.0,1.839093e+05,0.561391,0.513043,3251.121510,58.409084,3.599353e+05,6496.586382,115.875857,39.967603,0.430942,10.545435
min,24890.000000,2019.0,1.650000e+04,0.000000,-3.638000,0.000000,96.000000,8.000000e+03,0.000000,500.000000,0.967478,0.000000,5.000000
25%,62057.250000,2019.0,1.965000e+05,3.625000,0.076000,581.490000,360.000000,2.680000e+05,3720.000000,599.000000,60.474860,0.000000,31.000000
50%,99224.500000,2019.0,2.965000e+05,3.990000,0.390400,2596.450000,360.000000,4.180000e+05,5760.000000,699.000000,75.135870,0.000000,39.000000
75%,136391.750000,2019.0,4.365000e+05,4.375000,0.775400,4812.500000,360.000000,6.280000e+05,8520.000000,800.000000,86.184211,0.000000,45.000000
max,173559.000000,2019.0,3.576500e+06,8.000000,3.357000,60000.000000,360.000000,1.650800e+07,578580.000000,900.000000,7831.250000,1.000000,61.000000


In [5]:
# Análise simplificada de variáveis
target_col = "Status"

# Para variáveis categóricas, podemos usar o método `value_counts()` para calcular a odds de cada categoria
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

odds_df = pd.DataFrame(columns=['col', 'category', 'odds'])
for col in categorical_cols:
    # Para cada categoria, vamos ver a distribuição das classes
    for category in df[col].unique():
        subset = df[df[col] == category]
        odds = subset[target_col].value_counts(normalize=True)
        odds_df.loc[len(odds_df)] = {'col': col, 'category': category, 'odds': odds.get(1, None)}

C:\Users\lucas\AppData\Local\Temp\ipykernel_26320\3625242942.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns.tolist()


In [6]:
# Para avaliar variáveis numéricas, podemos calcular a média e o desvio padrão para cada classe
numerical_cols = df.select_dtypes(include=['number']).columns.tolist()

numerical_df = pd.DataFrame(columns=['col', 'mean_class_0', 'mean_class_1', 'std_class_0', 'std_class_1'])
for col in numerical_cols:
    means = df.groupby(target_col)[col].mean()
    stds = df.groupby(target_col)[col].std()
    numerical_df.loc[len(numerical_df)] = {
        'col': col,
        'mean_class_0': means.get(0, None),
        'mean_class_1': means.get(1, None),
        'std_class_0': stds.get(0, None),
        'std_class_1': stds.get(1, None)
    }

In [7]:
numerical_df

,col,mean_class_0,mean_class_1,std_class_0,std_class_1
0,ID,99182.699530,99352.313218,42925.111042,42894.457217
1,year,2019.000000,2019.000000,0.000000,0.000000
2,loan_amount,334990.774875,319275.184912,174916.570573,208576.810054
3,rate_of_interest,4.044931,4.350500,0.561356,0.495546
4,Interest_rate_spread,0.441656,NaN,0.513043,NaN
5,Upfront_charges,3227.328554,1565.237974,3251.673989,2299.820513
6,term,335.144592,335.112085,58.824650,57.120208
7,property_value,505606.066286,457786.009377,342784.462653,436255.032008
8,income,7204.014214,6231.806780,6201.339600,7247.649876
9,Credit_Score,699.523793,700.600344,115.674510,116.487189


In [8]:
odds_df.dropna(inplace=True)
odds_df.sort_values(by='odds', ascending=False, inplace=True)

odds_df

,col,category,odds
64,Security_Type,Indriect,1.0
32,construction_type,mh,1.0
37,Secured_by,land,1.0
43,credit_type,EQUI,0.999935
30,lump_sum_payment,lpsm,0.776596
25,Neg_ammortization,neg_amm,0.445965
40,total_units,3U,0.384224
11,loan_type,type2,0.345439
23,business_or_commercial,b/c,0.345439
39,total_units,2U,0.345295


In [9]:
# Seleção de variáveis
variables = [
    "age", # Categórica ordinal
    "credit_type", # Categórica nominal
    "Upfront_charges", # Numérica contínua
    "Credit_Score" # Numérica contínua
]


# Dataset reduzido que será usado para construção do modelo
df = df[variables + [target_col]]

# O objetivo desse exercício é replicar um  modelo de regressõa logística,
# por isso, vou ignorar questões relacionadas a tratamento de dados e
# feature engineering por enquanto.
df = df.dropna() # Não quero lidar com NAs

df.info()

<class 'pandas.DataFrame'>
Index: 108875 entries, 2 to 148669
Data columns (total 5 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   age              108875 non-null  str    
 1   credit_type      108875 non-null  str    
 2   Upfront_charges  108875 non-null  float64
 3   Credit_Score     108875 non-null  int64  
 4   Status           108875 non-null  int64  
dtypes: float64(1), int64(2), str(2)
memory usage: 5.0 MB


In [10]:
# One-hot-encoding para "credit_type"
df = pd.get_dummies(df, columns=['credit_type'], drop_first=True)

# Ordinal encoding para "age"
df['age'] = df['age'].map({
    '<25': 0,
    '25-34': 1,
    '35-44': 2,
    '45-54': 3,
    '55-64': 4,
    '65-74': 5,
    '>74': 6
})

# Converter todas as colunas para float
df = df.astype(float)

variables = list(df.columns)
variables.remove(target_col)

# Normalizar todas as variáveis
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

df[variables] = scaler.fit_transform(df[variables])

In [11]:
df.info()

<class 'pandas.DataFrame'>
Index: 108875 entries, 2 to 148669
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   age               108875 non-null  float64
 1   Upfront_charges   108875 non-null  float64
 2   Credit_Score      108875 non-null  float64
 3   Status            108875 non-null  float64
 4   credit_type_CRIF  108875 non-null  float64
 5   credit_type_EQUI  108875 non-null  float64
 6   credit_type_EXP   108875 non-null  float64
dtypes: float64(7)
memory usage: 6.6 MB


In [12]:
df.describe()

,age,Upfront_charges,Credit_Score,Status,credit_type_CRIF,credit_type_EQUI,credit_type_EXP
count,1.088750e+05,1.088750e+05,1.088750e+05,108875.0,1.088750e+05,1.088750e+05,1.088750e+05
mean,1.670714e-16,8.771246e-17,1.551610e-16,0.0,1.088574e-16,1.305245e-18,1.260867e-16
std,1.000005e+00,1.000005e+00,1.000005e+00,0.0,1.000005e+00,1.000005e+00,1.000005e+00
min,-2.194802e+00,-9.925175e-01,-1.724295e+00,0.0,-6.992350e-01,-3.030665e-03,-6.697728e-01
25%,-7.749030e-01,-8.127969e-01,-8.685246e-01,0.0,-6.992350e-01,-3.030665e-03,-6.697728e-01
50%,-6.495346e-02,-1.929257e-01,-4.110522e-03,0.0,-6.992350e-01,-3.030665e-03,-6.697728e-01
75%,6.449961e-01,4.883619e-01,8.689477e-01,0.0,1.430134e+00,-3.030665e-03,1.493044e+00
max,2.064895e+00,1.745960e+01,1.733362e+00,0.0,1.430134e+00,3.299606e+02,1.493044e+00


# 02. Construção do Modelo
Para construir o modelo, preciso das seguintes equações:
1. Função logística: Calcula a probabilidade estimada de "Default" para aquela observação. Essa função vai ser usada como 'predict' depois do 'fit' do modelo.
2. Função do vetor gradiente: Vai ser usada para 'fit' do modelo, mais especificamente seu output será usado para atualizar os pesos do modelo. A função do vetor gradiente será a derivada de log likelihood. O log é usado para previnir underflow e simplificar a derivação.

Também vou inicializar uma lista com os pesos do modelo, incluindo "beta 0" (termo independente).

In [17]:
# Gerar matriz de features (X) e vetor de target (y)
y = df[target_col].to_numpy()
X = df[variables].to_numpy()

# Definir lista que armazena os "betas" do modelo
ones = np.ones((len(X), 1)) # Nova coluna com 1 em todos os valores (vai servir para o termo independente)
X_b = np.hstack([ones, X])

weights = np.ones(X_b.shape[1]) # Vetor de pesos (betas)

# Definir função de probabilidade logística
def logistic_function(weights, x):
    z = x @ weights
    z = np.clip(z, a_min=-700, a_max=700) # Evitar overflow em np.exp
    return 1 / (1 + np.exp(-z))

# Definição da função do vetor gradiente
def gradient(weights, x, y):
    p = logistic_function(weights, x)
    return (x.T @ (p - y)) / len(y)

# Função de custo (log loss). Vai ser usada para avaliar o modelo durante treino
def log_loss(weights, x, y):
    p = logistic_function(weights, x)
    p = np.clip(p, a_min=1e-15, a_max=1 - 1e-15) # Evitar log(0) ou log (1)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

In [18]:
passo = 0.01

for epoch in range(1000):
    grad = gradient(weights, X_b, y)
    weights -= passo * grad

    if epoch % 100 == 0:
        loss = log_loss(weights, X_b, y)
        print(f"epoch={epoch}, loss={loss:.6f}")

epoch=0, loss=1.652940
epoch=100, loss=1.215689
epoch=200, loss=0.875314
epoch=300, loss=0.626438
epoch=400, loss=0.454590
epoch=500, loss=0.340106
epoch=600, loss=0.264274
epoch=700, loss=0.213030
epoch=800, loss=0.177204
epoch=900, loss=0.151201


# 03. Anotações Extras

Esse modelo não foi simples de entender. Também tive alguns erros enquanto criava o notebook.
1. Na prática, me parece que a regressão logística não depende de fato do log(odds) na etapa de treino/teste. A ideia do log(odds) é apenas interpretar o modelo de fato como um GLM, mas o treinamento é feito maximizando a log(likehood) (ou minimizando log-loss, nesse caso), e o teste é feito com métricas em cima da probabilidade predita (sigmoide). Em nenhum momento dessas etapas de fato usamos a log(odds)
2. O padrão em machine learning é minizar a função de perda, então geralmente não utilizamos a log(likelihood) no treinamento (a pesar de tecnicamente ser possível).
3. O log ainda assim é bastante importante no modelo, mas principalmente para facilitar as operações matemáticas intermediárias (a derivada principalmente), e para regularizar o eixo y em algumas situações.

Sobre os erros:
1. Diferentemente do modelo de árvore, os modelos lineares não vão lidar bem com valores grandes de parâmetros não normalizados. Sem a normalização o modelo não vai convergir.